> 📅 __Date: 2026-08-27__

# 📝 **Document Summarization with LangChain**

> **Goal:** Understand how to build a document-summarization pipeline with LangChain, why a normal LLM call can fail for long documents, how document loaders and `Document` objects work, and how the `stuff`, `map_reduce`, and `refine` summarization strategies process documents differently.

---

# 🗺️ **Document Summarization Roadmap**

**A useful way to understand LangChain summarization is:**

```text
LLM
 ↓
LLM Limitations
 ↓
Document
 ↓
Document Loader
 ↓
Document Object
 ↓
Long Document Problem
 ↓
Summarization Chain
 ↓
 ┌───────────────┬────────────────┬───────────────┐
 ↓               ↓                ↓
Stuff          Map-Reduce       Refine
 ↓               ↓                ↓
Summary        Summary           Summary
```

---

# 🤖 **Basic LLM Example**

Before working with documents, let's see how a normal LLM call works.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import OpenAI

model = OpenAI()

response = model.invoke(
    "What is the capital of France?"
)

print(response)



The capital of France is Paris.


**The basic flow is:**

```text
User Prompt
    ↓
LLM
    ↓
Generated Response
```

This works well for small inputs.

However, when we start working with **private data and large documents**, an LLM has several limitations.

---

# ⚠️ **Limitations of LLMs**

There are several important limitations that become especially visible when building document-based applications.

---

# 1️⃣ **Private Data**

> **LLMs are generally not aware of your application's private or internal data unless that information is explicitly provided to the model through the application.**

**For example:**

In [3]:
model.invoke(
    "What is the current salary structure for employees in our company?"
)

'\nThe current salary structure for employees in our company varies based on job title, level of experience, and location. Generally, our company offers competitive salaries that are in line with industry standards and are based on a combination of factors such as market demand, employee performance, and company budget. Our HR department regularly reviews and updates the salary structure to ensure that it remains fair and competitive. Additionally, employees may also be eligible for bonuses, benefits, and other forms of compensation depending on their role and performance.'

**The model may not know the answer if that information exists only in:**

```text
Your internal documents
Your database
Your internal website
Your organization
Your private communications
```

**Conceptually:**

```text
Private Data
     ❌
     ↓
Pretrained LLM
     ↓
No reliable access to private information
```

**Therefore, an application may need to provide private information through mechanisms such as:**

```text
Prompt Context
+
Retrieval
+
RAG
+
Database / Tool Access
```

> **An LLM does not automatically know the private data of your application.**

---

# 2️⃣ **Context Window**

> **Context Window = The maximum amount of token context that a model can process for a request / generation setup.**

**Conceptually:**

```text
System Instructions
       +
User Input
       +
Conversation History
       +
Retrieved Context
       +
Output
       ↓
   Context Window
```

If the input becomes too large, the model may not be able to process the complete request.

**A simplified constraint is:**

$$
\text{Input Tokens}
+
\text{Output Tokens}
\leq
\text{Model Context Limit}
$$

The exact limit depends on the model.

---

# 3️⃣ **LLMs are Stateless**

> **LLMs are stateless by default: each independent model invocation does not automatically remember previous interactions.**

**For example:**

In [4]:
model.invoke(
    "What is the capital of France?"
)

'\n\nThe capital of France is Paris.'

**Then:**

In [5]:
model.invoke(
    "Tell me more about it?"
)

'\n\nThe term "tell me more" is often used to prompt someone to provide additional information or details about a particular topic or situation. It is a way of expressing interest and curiosity in learning more about something. This phrase can be used in a variety of contexts, such as in a conversation with a friend, during a job interview, or in a therapy session. It can also be used as a way to encourage someone to open up and share more about their thoughts, feelings, or experiences. Overall, "tell me more" is a simple yet powerful way to show interest and engage in deeper communication.'

The second request is a separate invocation.

The model does not automatically receive the earlier question unless the application provides the conversation history.

**Conceptually:**

```text
Request 1
   ↓
LLM
   ↓
Response 1


Request 2
   ↓
LLM
   ↓
Response 2
```

**There is no automatic connection:**

```text
Request 1
   X
Request 2
```

**Instead, the application can maintain state:**

```text
Previous Messages
      +
New Message
      ↓
LLM
      ↓
New Response
```

> **Memory in an LLM application is usually an application-level capability, not something automatically retained by every model call.**

---

# 4️⃣ **Knowledge Cutoff**

> **Knowledge Cutoff = A point in time beyond which the model's training data may not contain information.**

A model trained on data available up to a certain period cannot automatically know every event that happened afterward.

**For example:**

In [6]:
model.invoke(
    "Who is Abhijeet Dipake?"
)

'\n\nThere is not enough information available to determine who Abhijeet Dipake is. It is possible that he is a private individual or a relatively unknown public figure.'

**or:**

In [7]:
model.invoke(
    "Who is the CM of Maharashtra?"
)

'\n\nThe current Chief Minister of Maharashtra is Uddhav Thackeray.'

**The reliability of the response depends on:**

```text
Training Data
+
Knowledge Cutoff
+
Currentness of the Question
+
Whether External Information is Provided
```

**For current information, applications can use:**

```text
Web Search
APIs
Databases
RAG
Tools
```

---

# 5️⃣ **Hallucination**

> **Hallucination = When an LLM generates information that is false, unsupported, or misleading.**

**For example:**

In [8]:
model.invoke(
    "What is LangChain?"
)

'\n\nLangChain is a decentralized platform that aims to connect language learners with native speakers for real-time language practice and cultural exchange. It utilizes blockchain technology to provide secure and transparent interactions between users and to incentivize participation through its own cryptocurrency. The platform also offers language learning resources and tools, as well as opportunities for users to earn rewards for contributing to the community.'

The model may produce a confident answer, but confidence does not guarantee factual correctness.

**Conceptually:**

```text
Question
   ↓
LLM
   ↓
Plausible Response
   ↓
Could be:
✅ Correct
⚠️ Partially Correct
❌ Incorrect / Unsupported
```

Therefore:

> **LLM output should not automatically be treated as ground truth.**

For document-based applications, one important way to reduce unsupported answers is to provide relevant source information to the model.

---

# 🧠 **Why Document Summarization?**

**Suppose we have a large PDF such as:**

```text
NIPS-2017-attention-is-all-you-need-Paper.pdf
```

**We want:**

```text
Large Document
      ↓
LLM
      ↓
Concise Summary
```

The challenge is that the whole document may be too large to send to the model in one prompt.

**This leads to the core problem:**

```text
Large Document
      ↓
Too Many Tokens
      ↓
Context Window / Request Limit
      ↓
❌ Cannot simply stuff everything into one prompt
```

LangChain provides document-processing abstractions that can help us handle this problem.

---

# 📚 **Project: Summarize a Document**

### **Input Document**

```text
NIPS-2017-attention-is-all-you-need-Paper.pdf
```

### **Goal**

```text
PDF
 ↓
Extract Content
 ↓
Process Documents
 ↓
Generate Summary
```

---

# 📦 **Document Loaders**

> **Document Loader = A component that loads and extracts content from a source into LangChain `Document` objects.**

Different sources can require different loaders.

**Examples:**

```text
PDF
 ↓
PDF Loader

Web Page
 ↓
Web Loader

CSV
 ↓
CSV Loader

Text File
 ↓
Text Loader
```

The loader abstracts the source-specific extraction process.

---

# 🧩 **Why Document Loaders?**

**Without a loader, the application may need to manually handle:**

```text
Opening the file
Reading the file
Extracting text
Handling pages
Capturing metadata
Creating a standard structure
```

A document loader gives us a common representation.

**Conceptually:**

```text
Different Sources
 ┌───────┬────────┬─────────┐
 ↓       ↓        ↓         ↓
 PDF    HTML      CSV       TXT
 └───────┴────────┴─────────┘
             ↓
       Document Loader
             ↓
      LangChain Documents
```

---

# 📦 **Installing the Required Packages**

For PDF loading, the required packages depend on the loader and environment.

**A typical setup is:**

```python
%pip install -U langchain-community pypdf
```

> The exact loader-specific dependency can vary by document loader and LangChain version.

**The important idea is:**

```text
langchain-community
        +
loader-specific library
        ↓
Document Loader
```

**Examples of loader-specific libraries can include:**

```text
pypdf
PyPDF2
unstructured
...
```

---

# 📄 **PyPDFLoader**

**For the example PDF:**

In [9]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

C:\Users\KP\AppData\Local\Temp\ipykernel_5860\1550334054.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**At this point:**

```text
PDF File
  ↓
PyPDFLoader
```

The loader object is responsible for reading the document.

---

# ▶️ **Loading the Document**

In [10]:
docs = loader.load()

**The important point is:**

> `load()` returns a **list of `Document` objects**.

**So:**

```python
type(docs)
```

**will conceptually be:**

```text
list
```

**And:**

```python
type(docs[0])
```

**will conceptually be:**

```text
Document
```

---

# 🧩 **Document Object**

> **Document = A standardized LangChain object containing extracted content and associated metadata.**

A `Document` commonly contains two important pieces of information:

```text
Document
 ├── page_content
 └── metadata
```

---

# 1️⃣ **page_content**

> **`page_content` = The extracted textual content of the document chunk / page.**

**Example:**

```python
docs[0].page_content
```

This returns the text extracted from the first loaded document item.

**Conceptually:**

```text
PDF Page
   ↓
Text Extraction
   ↓
page_content
```

---

# 2️⃣ **metadata**

> **`metadata` = Information describing the source or origin of the document content.**

**Metadata can contain information such as:**

```text
Source Path
Page Number
Document Identifier
Other Loader-Specific Information
```

**Conceptually:**

```text
Document
 ├── page_content
 │      ↓
 │   Actual Text
 │
 └── metadata
        ↓
     Source Information
```

---

# 🔍 **Inspecting the Loaded Documents**

In [11]:
type(docs)

list

In [12]:
type(docs[0])

langchain_core.documents.base.Document

In [13]:
docs[0].page_content

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantly

**And metadata can be inspected with:**

In [14]:
docs[0].metadata

{'producer': 'PyPDF2',
 'creator': 'PyPDF',
 'creationdate': '',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'publisher': 'Curran Associates, Inc.',
 'language': 'en-US',
 'created': '2017',
 'eventtype': 'Poster',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-

This is useful for understanding exactly what the loader produced.

---

# 🗂️ **Document Structure**

**For a multi-page PDF:**

```text
PDF
 │
 ├── Page 1 → Document
 │              ├── page_content
 │              └── metadata
 │
 ├── Page 2 → Document
 │              ├── page_content
 │              └── metadata
 │
 ├── Page 3 → Document
 │              ├── page_content
 │              └── metadata
 │
 └── ...
```

**Therefore:**

In [15]:
docs

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

is a collection of loaded `Document` objects.

---

# 🧪 **Reading All Document Text**

**A simple manual approach is:**

In [16]:
text = ""

for page in docs:
    text += page.page_content

print(text)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model a

**The flow is:**

```text
Document 1
   ↓
page_content
   ↓
Document 2
   ↓
page_content
   ↓
Document 3
   ↓
...
   ↓
One Large Text String
```

This works for inspection, but it creates another problem.

---

# ⚠️ **The Long Document Problem**

**Now suppose we build:**

```python
prompt = f"Summarize the following text: {text}"
```

Then the whole document is sent to the model in one request.

**Conceptually:**

```text
Entire PDF
    ↓
Extract Entire Text
    ↓
Insert Entire Text into Prompt
    ↓
LLM
```

**For a sufficiently large PDF:**

```text
Document Tokens
      ↓
Prompt Tokens
      ↓
Context Limit
      ↓
❌ Request Too Large
```

---

# 🚨 **Example Context-Length Error**

**A model may return an error similar to:**

```text
BadRequestError: Error code: 400

This model's maximum context length is 4097 tokens,
however you requested 8515 tokens
```

The exact numbers depend on the model and request.

**The important lesson is:**

> **The full document cannot always be sent to the model in a single request.**

---

# 🧠 **Why Does This Happen?**

**Suppose:**

```text
Document = 8,259 input tokens
Output = 256 tokens
```

**Then approximately:**

$$
8,259 + 256 = 8,515
$$

**If the model supports only:**

$$
4,097 \text{ tokens}
$$

**then:**

$$
8,515 > 4,097
$$

Therefore the request exceeds the model's maximum context length.

---

# 🧮 **Context Window Constraint**

**A simplified condition is:**

$$
T_{\text{input}} + T_{\text{output}}
\leq
T_{\text{context}}
$$

**where:**

```text
T_input
→ Input tokens

T_output
→ Requested output tokens

T_context
→ Model context limit
```

**When:**

$$
T_{\text{input}} + T_{\text{output}}
>
T_{\text{context}}
$$

the request cannot be processed under that limit.

---

# 💡 **Solution: Summarization Chains**

Instead of sending the entire document in one prompt, we can use a summarization chain.

```text
Large Document
      ↓
Summarization Strategy
      ↓
Smaller Processing Steps
      ↓
Final Summary
```

This is where:

```text
load_summarize_chain()
```

becomes useful.

---

# 🔗 **`load_summarize_chain`**

In [17]:
from langchain_classic.chains import load_summarize_chain

**Create a summarization chain:**

In [18]:
chain = load_summarize_chain(model)

**Then:**

```python
summary = chain.invoke(docs)
```

**Conceptually:**

```text
List of Documents
       ↓
Summarization Chain
       ↓
Summary
```

---

# 🧩 **Chain Type**

> **`chain_type` = A configuration that determines how a summarization chain processes the input documents.**

Different chain types use different document-processing strategies.

**The three important strategies are:**

```text
1. Stuff
2. Map Reduce
3. Refine
```

---

# 🥇 **1. Stuff Chain**

> **Stuff = Combine the available document content into one prompt and send it to the model in a single call.**

This is the simplest summarization strategy.

**Conceptually:**

```text
Document 1 ─┐
Document 2 ─┤
Document 3 ─┤
Document 4 ─┤
     ...     ┤
Document N ──┘
       ↓
 Combine / "Stuff"
       ↓
    One Prompt
       ↓
      LLM
       ↓
    Summary
```

---

# 🖼️ **Stuff Chain Architecture**

<div align="center">

<img src="assets/Chain_type_Stuff.png" width="800" alt="LangChain Stuff Summarization Chain">

<p><em>Figure: Stuff summarization chain — document content is combined into a single model prompt.</em></p>

</div>

---

# 🧪 **Stuff Chain Code**

```python
from langchain_classic.chains import load_summarize_chain

chain = load_summarize_chain(
    model,
    chain_type="stuff"
)
```

**`stuff` is the default chain type in the example:**

```python
chain = load_summarize_chain(
    model
)
```

**is equivalent in concept to:**

```python
chain = load_summarize_chain(
    model,
    chain_type="stuff"
)
```

**Then:**

```python
summary = chain.invoke(docs)

print(summary["output_text"])
```

> ⚠️ **Note:** We are not executing this Stuff-chain example for this particular PDF. The document contains enough pages/chunks that combining all of them into a single prompt would exceed the LLM's available context window. Therefore, this example is shown only to understand how the Stuff chain works; running it with this document can result in a context-length error.

<details>
<summary>🔴 <strong>Context Window Error</strong></summary>

```text
BadRequestError: Error code: 400

This model's maximum context length is 4097 tokens,
however you requested 8533 tokens
(8277 in your prompt; 256 for the completion).

Please reduce your prompt or completion length.
```
</details>

---

# 🔍 **How Stuff Works**

**Suppose we have:**

```text
Document 1
Document 2
Document 3
```

**The chain effectively creates:**

```text
Prompt
=
Instructions
+
Document 1
+
Document 2
+
Document 3
```

**and then:**

```text
Prompt
  ↓
LLM
  ↓
Summary
```

---

# ✅ **Advantages of Stuff**

```text
Simple
Fast for small inputs
Single summarization step
Easy to understand
```

---

# ⚠️ **Limitations of Stuff**

The entire input has to fit within the model's context constraints.

```text
Large Documents
      ↓
All content combined
      ↓
Context Window
      ↓
Potentially Too Large
```

**Therefore:**

> **Stuff is best suited to documents or document sets that fit within the available context.**

---

# 🥈 **2. Map-Reduce Chain**

> **Map-Reduce = Summarize document pieces independently, then combine those intermediate summaries into a final summary.**

This approach is useful when the complete document cannot fit into one prompt.

**Conceptually:**

```text
Document 1 ──→ LLM ──→ Summary 1
Document 2 ──→ LLM ──→ Summary 2
Document 3 ──→ LLM ──→ Summary 3
Document 4 ──→ LLM ──→ Summary 4
                         │
                         ↓
                 Combine Summaries
                         ↓
                        LLM
                         ↓
                  Final Summary
```

---

# 🖼️ **Map-Reduce Chain Architecture**

<div align="center">

<img src="assets/chain_type_map_reduce.png" width="800" alt="LangChain Map Reduce Summarization Chain">

<p><em>Figure: Map-Reduce summarization — summarize document pieces independently, then combine the intermediate summaries.</em></p>

</div>

---

# 🧭 **Map Step**

Each document or document chunk is processed independently.

```text
Document 1 → Summary 1
Document 2 → Summary 2
Document 3 → Summary 3
```

This is the **Map** stage.

**Conceptually:**

$$
S_i = f(D_i)
$$

**where:**

```text
D_i
→ Individual document / chunk

f
→ Summarization operation

S_i
→ Intermediate summary
```

---

# 🔗 **Reduce Step**

The intermediate summaries are then combined.

```text
Summary 1 ─┐
Summary 2 ─┤
Summary 3 ─┤
Summary 4 ─┘
     ↓
Combine
     ↓
LLM
     ↓
Final Summary
```

**Conceptually:**

$$
S_{\text{final}}
=
g(S_1,S_2,\ldots,S_n)
$$

where `g` generates the final summary from the intermediate summaries.

---

# 🧪 **Map-Reduce Code**

In [19]:
chain = load_summarize_chain(
    model,
    chain_type="map_reduce"
)

summary = chain.invoke(docs)

print(summary["output_text"])



The article introduces the Transformer model, a new neural network architecture that uses attention mechanisms instead of recurrence or convolution. Experiments show that the Transformer outperforms previous models in machine translation tasks while being more parallelizable and requiring less training time. The authors credit various team members for their contributions to the model. The article also discusses techniques used in the Transformer, such as Scaled Dot-Product Attention and Multi-Head Attention, as well as the use of self-attention and positional encodings. The Transformer sets a new state-of-the-art BLEU score and future research plans involve applying attention-based models to other tasks. References are provided for further research on neural networks and deep learning.


**The important difference from Stuff is:**

```text
Stuff
→ Entire content together

Map-Reduce
→ Process pieces separately
→ Combine intermediate summaries
```

---

# ✅ **Advantages of Map-Reduce**

```text
Handles larger document collections
Reduces the size of each individual model input
Can process independent document pieces
Naturally fits distributed / parallel processing patterns
```

---

# ⚠️ **Limitations of Map-Reduce**

```text
Multiple model calls
Higher cost
More complex workflow
Intermediate summaries may lose detail
Final summary depends on the quality of both map and reduce stages
```

**A key trade-off is:**

$$
\text{Scalability}
\uparrow
\quad\Longleftrightarrow\quad
\text{Number of Model Calls}
\uparrow
$$

---

# 🥉 **3. Refine Chain**

> **Refine = Build an initial summary from one document / chunk, then iteratively update that summary using subsequent documents / chunks.**

**The basic idea is:**

```text
Document 1
   ↓
LLM
   ↓
Initial Summary
   ↓
Document 2
   ↓
LLM
   ↓
Refined Summary
   ↓
Document 3
   ↓
LLM
   ↓
More Refined Summary
   ↓
...
```

---

# 🖼️ **Refine Chain Architecture**

<div align="center">

<img src="assets/Chain_type_refine.png" width="800" alt="LangChain Refine Summarization Chain">

<p><em>Figure: Refine summarization — an existing summary is progressively updated with additional document content.</em></p>

</div>

---

# 🧭 **How Refine Works**

**Suppose there are four documents:**

```text
D1
D2
D3
D4
```

**First:**

```text
D1
 ↓
LLM
 ↓
S1
```

**Then:**

```text
S1 + D2
   ↓
  LLM
   ↓
  S2
```

**Then:**

```text
S2 + D3
   ↓
  LLM
   ↓
  S3
```

**Finally:**

```text
S3 + D4
   ↓
  LLM
   ↓
Final Summary
```

**Mathematically:**

$$
S_1 = f(D_1)
$$

$$
S_2 = f(S_1,D_2)
$$

$$
S_3 = f(S_2,D_3)
$$

**and generally:**

$$
S_i = f(S_{i-1},D_i)
$$

---

# 🧪 **Refine Chain Code**

In [20]:
chain = load_summarize_chain(
    model,
    chain_type="refine"
)

summary = chain.invoke(docs)

print(summary["output_text"])



The Transformer is a new attention-based network architecture that eliminates the need for recurrent or convolutional neural networks and relies entirely on self-attention. Developed by researchers from Google and the University of Toronto, it achieves state-of-the-art results on translation tasks and allows for more parallelization and faster training time. The model is composed of stacked encoder and decoder layers with residual connections and layer normalization. The attention function used in the model is "Scaled Dot-Product Attention", which computes the dot product of queries and keys to generate weighted values. The Transformer utilizes multi-head attention, allowing for more efficient processing by jointly attending to information from different representation subspaces at different positions. Additionally, the model employs masking in the decoder stack to prevent positions from attending to subsequent positions, ensuring more accurate predictions. The encoder-decoder and se

---

# ✅ **Advantages of Refine**

```text
Can preserve an evolving summary
Can incorporate information from later documents
Useful when the summary should progressively absorb new details
```

---

# ⚠️ **Limitations of Refine**

**Because the summary is updated sequentially:**

```text
D1
 ↓
D2
 ↓
D3
 ↓
D4
 ↓
...
```

the workflow requires multiple model calls.

**Therefore:**

```text
More Documents
      ↓
More Refinement Steps
      ↓
More Model Calls
      ↓
Higher Latency / Cost
```

It can also depend strongly on the quality of the intermediate summary because each later stage works from the current summary.

---

# 🆚 **Stuff vs Map-Reduce vs Refine**

| Feature | Stuff | Map-Reduce | Refine |
|---|---|---|---|
| **Basic Idea** | Combine everything into one prompt | Summarize pieces, then combine | Iteratively update a summary |
| **Model Calls** | Usually one main call | Multiple map calls + reduce | Multiple sequential calls |
| **Large Documents** | Limited by context | Better suited | Better suited |
| **Parallelism** | Limited | Map stage can be parallelized conceptually | Mostly sequential |
| **Complexity** | Low | Medium | Medium |
| **Latency** | Low for small inputs | Can be reduced through parallel map work, but depends on implementation | Sequential calls can increase latency |
| **Information Flow** | All content at once | Intermediate summaries | Current summary + next document |
| **Use Case** | Small documents | Large document collections | Progressive / iterative summarization |

---

# 🧠 **Memory Trick**

```text
STUFF
→ Stuff everything into one prompt.

MAP-REDUCE
→ Map: Summarize pieces.
→ Reduce: Combine summaries.

REFINE
→ Start with a summary.
→ Keep refining it.
```

---

# 📊 **Visual Comparison**

```text
STUFF
──────────────────────────────

D1 ─┐
D2 ─┤
D3 ─┤
D4 ─┤
... │
Dn ─┘
 ↓
One Prompt
 ↓
LLM
 ↓
Summary
```

```text
MAP-REDUCE
──────────────────────────────

D1 → Summary 1 ─┐
D2 → Summary 2 ─┤
D3 → Summary 3 ─┼→ Combine → LLM → Final Summary
D4 → Summary 4 ─┤
...              │
Dn → Summary n ─┘
```

```text
REFINE
──────────────────────────────

D1 → LLM → S1
             ↓
D2 ─────────→ LLM → S2
                       ↓
D3 ───────────────────→ LLM → S3
                                 ↓
D4 ─────────────────────────────→ LLM
                                         ↓
                                    Final Summary
```

---

# 🧮 **Complexity Intuition**

**Let:**

```text
N = Number of document pieces
```

### **Stuff**

**The model performs one main summarization call:**

$$
\text{Calls} \approx 1
$$

subject to the complete input fitting within the context constraints.

### **Map-Reduce**

Conceptually:

$$
\text{Calls}
\approx
N + 1
$$

**where:**

```text
N
→ Map calls

1
→ Reduce call
```

The actual implementation can vary.

### **Refine**

**Conceptually:**

$$
\text{Calls}
\approx
N
$$

because each piece participates in an iterative refinement process.

> **These are conceptual counts for understanding the strategies; implementation details can change the exact number of calls.**

---

# ⚖️ **Choosing the Right Chain Type**

**A simple decision framework:**

```text
Does the entire document fit comfortably
within the model's context?
            │
        ┌───┴───┐
       YES      NO
        ↓        ↓
      STUFF   Need multi-step processing
                 │
          ┌──────┴──────┐
          ↓             ↓
      MAP-REDUCE      REFINE
          │             │
          ↓             ↓
   Independent       Progressive
   summaries         refinement
```

**Another practical view:**

```text
Small Document
    ↓
Stuff

Large Collection
    ↓
Map-Reduce

Progressive / Iterative Summary
    ↓
Refine
```

---

# 🔄 **Complete Summarization Pipeline**

**A complete document-summarization application looks like:**

```text
PDF
 ↓
Document Loader
 ↓
List[Document]
 ↓
Summarization Chain
 ↓
 ┌───────────────┬────────────────┬───────────────┐
 ↓               ↓                ↓
Stuff          Map-Reduce       Refine
 ↓               ↓                ↓
Summary        Summary           Summary
```

---

# 🧩 **End-to-End Example — Stuff**

```python
from langchain_openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import load_summarize_chain

# 1. Create the model
model = OpenAI()

# 2. Load the PDF
loader = PyPDFLoader(
    "NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

# 3. Load documents
docs = loader.load()

# 4. Create summarization chain
chain = load_summarize_chain(
    model,
    chain_type="stuff"
)

# 5. Generate summary
summary = chain.invoke(docs)

# 6. Display summary
print(summary["output_text"])
```

**Flow:**

```text
PDF
 ↓
PyPDFLoader
 ↓
docs
 ↓
Stuff Chain
 ↓
LLM
 ↓
output_text
```

> ⚠️ **Note:** We are not executing this Stuff-chain example for this particular PDF. The document contains enough pages/chunks that combining all of them into a single prompt would exceed the LLM's available context window. Therefore, this example is shown only to understand how the Stuff chain works; running it with this document can result in a context-length error.

<details>
<summary>🔴 <strong>Context Window Error</strong></summary>

```text
BadRequestError: Error code: 400

This model's maximum context length is 4097 tokens,
however you requested 8533 tokens
(8277 in your prompt; 256 for the completion).

Please reduce your prompt or completion length.
```
</details>

---

# 🧩 **End-to-End Example — Map-Reduce**

In [22]:
from langchain_openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import load_summarize_chain

model = OpenAI()

loader = PyPDFLoader(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

docs = loader.load()

chain = load_summarize_chain(
    model,
    chain_type="map_reduce"
)

summary = chain.invoke(docs)

print(summary["output_text"])



The paper "Attention Is All You Need" introduces the Transformer model, which uses only attention mechanisms and outperforms traditional models in machine translation tasks. The authors, from Google and the University of Toronto, each contributed to the development of the model. The article explains the architecture of the Transformer, which includes self-attention and feed-forward layers. Multi-Head Attention is used to focus on different aspects of the input data. The paper also compares different types of layers and discusses positional encodings. The Transformer achieves better results and is more efficient compared to previous models. The authors plan to apply it to other tasks and provide the code for use. The article also provides references for various deep learning techniques and architectures used in natural language processing tasks.


**Flow:**

```text
PDF
 ↓
Documents
 ↓
Map
 ↓
Individual Summaries
 ↓
Reduce
 ↓
Final Summary
```

---

# 🧩 **End-to-End Example — Refine**

In [23]:
from langchain_openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import load_summarize_chain

model = OpenAI()

loader = PyPDFLoader(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

docs = loader.load()

chain = load_summarize_chain(
    model,
    chain_type="refine"
)

summary = chain.invoke(docs)

print(summary["output_text"])



The paper introduces the Transformer, an advanced network architecture that uses attention mechanisms for sequence transduction models. It is developed by a team from Google Brain, Google Research, and the University of Toronto. The model utilizes scaled dot-product attention, multi-head attention, and a parameter-free position representation, resulting in state-of-the-art translation quality after only 12 hours of training on eight P100 GPUs. The Transformer surpasses previous models in terms of efficiency and performance, as shown in Table 3. The paper also discusses the use of multi-head attention and positional encoding, and compares self-attention layers to recurrent and convolutional layers. With a new state-of-the-art BLEU score of 28.4 on the WMT 2014 English-to-German translation task, the Transformer sets a new standard in machine translation.


**Flow:**

```text
PDF
 ↓
Documents
 ↓
Initial Summary
 ↓
Refine with Next Document
 ↓
Refine Again
 ↓
...
 ↓
Final Summary
```

---

# 🧪 **Inspecting the Result**

**The example uses:**

```python
summary = chain.invoke(docs)
```

**and then:**

```python
print(summary["output_text"])
```

**The result can be treated conceptually as:**

```text
Chain Result
     ↓
Output Dictionary
     ↓
"output_text"
     ↓
Final Summary
```

The exact output structure depends on the chain API and version being used.

---

# 🧠 **Why the Three Strategies Exist**

There is no single best summarization strategy for every document.

The strategies exist because document size and information flow can differ.

```text
Small Input
    ↓
Stuff
```

```text
Large Input
    ↓
Map-Reduce
```

```text
Sequential / Progressive Summarization
    ↓
Refine
```

**The core engineering problem is:**

> **How can we summarize content while respecting the model's context limits and balancing quality, cost, and latency?**

---

# 💰 **Cost Perspective**

Suppose a document is divided into `N` pieces.

### **Stuff**

```text
One large model request
```

### **Map-Reduce**

```text
N map requests
+
1 reduce request
```

### **Refine**

```text
Initial request
+
Sequential refinement requests
```

Therefore, a multi-step strategy may increase the number of model calls.

**A simplified cost intuition is:**

$$
\text{Total Cost}
\approx
\sum_i
\text{Input Cost}_i
+
\sum_i
\text{Output Cost}_i
$$

For API-based models, every additional model call can contribute to cost and latency.

---

# ⚡ **Latency Perspective**

**A simplified latency view is:**

$$
\text{Total Latency}
\approx
\text{Network}
+
\text{Model Processing}
+
\text{Workflow Overhead}
$$

**For sequential refinement:**

```text
Call 1
 ↓
Call 2
 ↓
Call 3
 ↓
...
```

latency can accumulate across the sequence.

**For map-reduce:**

```text
Map 1 ─┐
Map 2 ─┤
Map 3 ─┤ → Reduce
Map 4 ─┘
```

the independent map stage can conceptually be parallelized, depending on the implementation and available infrastructure.

---

# 📦 **Why the Document Abstraction Matters**

LangChain's `Document` object provides a consistent representation.

**Instead of writing separate downstream logic for:**

```text
PDF page
Web page
CSV row
Text file
```

**the application can work with:**

```text
Document
 ├── page_content
 └── metadata
```

This abstraction is useful beyond summarization.

**It is also commonly used in:**

```text
RAG
Retrieval
Vector Stores
Semantic Search
Question Answering
Document Processing
```

---

# 🏗️ **Document Processing Mental Model**

```text
Raw Source
   ↓
Loader
   ↓
Document Objects
   ↓
Optional Splitting / Processing
   ↓
Model / Retriever / Chain
   ↓
Application Output
```

**For summarization:**

```text
Raw PDF
   ↓
PyPDFLoader
   ↓
Document Objects
   ↓
Summarization Chain
   ↓
Summary
```

---

# ⚠️ **Important: Loading is Not Summarization**

The loader only extracts content.

```text
Loader
→ Extracts / Loads Content
```

**It does not automatically:**

```text
Understand Content
Summarize Content
Reason About Content
Answer Questions
```

Those tasks are handled by the model / chain.

**Therefore:**

```text
PDF
 ↓
Loader
 ↓
Documents
 ↓
Summarization Chain
 ↓
LLM
 ↓
Summary
```

---

# 🆚 **Loader vs Summarization Chain**

| Component | Responsibility |
|---|---|
| **Document Loader** | Load and extract content |
| **Document** | Store content + metadata |
| **Summarization Chain** | Process documents for summarization |
| **LLM** | Generate summary text |

**Memory trick:**

```text
Loader
→ Get the data

Document
→ Hold the data

Chain
→ Process the data

LLM
→ Generate the summary
```

---

# 🔍 **Why Not Just Increase the Context Window?**

A larger context window can help, but it does not eliminate the need for document-processing strategies.

**A practical summarization system may still need to consider:**

```text
Context Limit
+
Cost
+
Latency
+
Information Density
+
Retrieval
+
Document Size
```

**Therefore:**

> **Document processing strategies are useful even when models support large context windows.**

---

# 🧠 **Summarization Strategy Decision Table**

| Situation | Preferred Strategy |
|---|---|
| Small document | **Stuff** |
| Entire content fits in context | **Stuff** |
| Large document collection | **Map-Reduce** |
| Need independent processing of pieces | **Map-Reduce** |
| Need progressive summary updates | **Refine** |
| Need simplest implementation | **Stuff** |
| Need iterative incorporation of information | **Refine** |

---

# 🏁 **Complete Mental Model**

```text
                     DOCUMENT SUMMARIZATION
                              │
                              ↓
                           SOURCE
                              │
                        ┌─────┴─────┐
                        ↓           ↓
                       PDF      Other Sources
                        │           │
                        └─────┬─────┘
                              ↓
                      DOCUMENT LOADER
                              ↓
                    List[Document Objects]
                              │
                 ┌────────────┼────────────┐
                 ↓            ↓            ↓
            page_content   metadata     source
                 │
                 ↓
          LARGE DOCUMENT PROBLEM
                 │
                 ↓
          CONTEXT LIMITATION
                 │
                 ↓
       SUMMARIZATION STRATEGY
                 │
      ┌──────────┼───────────┐
      ↓          ↓           ↓
    STUFF    MAP-REDUCE    REFINE
      ↓          ↓           ↓
     LLM       LLMs      Sequential LLMs
      ↓          ↓           ↓
   Summary    Summary     Summary
```

---

# 📋 **Quick Revision**

```text
LLM
→ Language model used for generation

Private Data
→ Information the model does not automatically know

Context Window
→ Maximum token context available for a request

Stateless
→ Separate model calls do not automatically remember earlier calls

Knowledge Cutoff
→ Training-data time boundary

Hallucination
→ False / unsupported generated information

Document Loader
→ Loads source data into LangChain Document objects

PyPDFLoader
→ Loads PDF content into Documents

Document
→ Standard object containing content and metadata

page_content
→ Extracted document text

metadata
→ Source / descriptive information associated with the Document

Summarization Chain
→ Workflow for generating summaries from Documents

chain_type
→ Controls how documents are processed

Stuff
→ Put document content into one prompt

Map-Reduce
→ Summarize pieces, then combine summaries

Refine
→ Iteratively update an existing summary

Map
→ Independent processing of document pieces

Reduce
→ Combine intermediate summaries

Refine
→ Update summary using additional content

output_text
→ Textual summary returned by the chain result
```

---

# 🆚 **Memory Trick: The Three Chains**

```text
STUFF
→ Everything together

MAP-REDUCE
→ Many summaries → One summary

REFINE
→ One summary → Keep improving it
```

---

# 🎯 **Interview-Friendly Explanation**

> **If an interviewer asks: "How do you summarize a large PDF using LangChain?"**

**A natural answer is:**

```text
First, I use a document loader such as PyPDFLoader
to extract the PDF into LangChain Document objects.

Each Document contains page_content and metadata.

If I try to concatenate the entire PDF and send it
to the LLM in a single prompt, I can hit the model's
context-window limit.

So I use a summarization chain.

For smaller documents, I can use the Stuff strategy,
where all the content is sent together.

For larger document collections, Map-Reduce can summarize
individual pieces first and then combine those summaries.

Refine takes a different approach: it creates an initial
summary and progressively updates it using subsequent content.
```

---

# 🧠 **Important Interview Differences**

### **Stuff**

```text
All Documents
     ↓
One Prompt
     ↓
LLM
     ↓
Summary
```

### **Map-Reduce**

```text
Documents
    ↓
Independent Summaries
    ↓
Combine
    ↓
LLM
    ↓
Final Summary
```

### **Refine**

```text
Document 1
    ↓
Initial Summary
    ↓
+ Document 2
    ↓
Refined Summary
    ↓
+ Document 3
    ↓
Refined Summary
    ↓
...
```

---

# ⚠️ **Common Beginner Mistakes**

## **1. Sending the Entire PDF as One Prompt**

```python
text = ""

for page in docs:
    text += page.page_content

prompt = f"Summarize the following text: {text}"

model.invoke(prompt)
```

**This can fail for large documents because:**

```text
Input Tokens
+
Output Tokens
>
Context Limit
```

---

## **2. Thinking `load()` Returns Raw Strings**

```python
docs = loader.load()
```

The result is a collection of `Document` objects.

**Use:**

```python
docs[0].page_content
```

to access the extracted text.

---

## **3. Confusing `page_content` with `metadata`**

```text
page_content
→ Actual extracted text

metadata
→ Information about the source / document
```

---

## **4. Assuming Stuff Always Handles Large Documents**

Stuff places the document content into one prompt.

**Therefore:**

```text
Large Document
      ↓
Stuff
      ↓
Context Limit
      ↓
Potential Failure
```

---

## **5. Assuming Map-Reduce and Refine Work the Same Way**

They are different.

```text
Map-Reduce
→ Independent intermediate summaries
→ Then combine

Refine
→ Sequentially update one evolving summary
```

---

## **6. Ignoring Model Calls and Cost**

Multi-step summarization can require many model calls.

```text
More Calls
   ↓
Potentially More Cost
   ↓
Potentially More Latency
```

---

## **7. Copying Legacy LangChain Code Without Checking Version**

**The example uses:**

```python
from langchain_classic.chains import load_summarize_chain
```

This is a legacy/classic-style API.

LangChain's package organization and APIs evolve over time, **so when starting a new project:**

> **Check the current LangChain documentation for the recommended summarization workflow and package.**

The conceptual strategies remain important even when the exact API changes.

---

# 🔬 **Comparing the Three Strategies with One Document**

**Suppose the PDF contains:**

```text
Page 1
Page 2
Page 3
Page 4
```

### **Stuff**

```text
Page 1 ─┐
Page 2 ─┤
Page 3 ─┤
Page 4 ─┘
   ↓
One Prompt
   ↓
LLM
   ↓
Summary
```

### **Map-Reduce**

```text
Page 1 → Summary 1
Page 2 → Summary 2
Page 3 → Summary 3
Page 4 → Summary 4
                  ↓
             Combine
                  ↓
                 LLM
                  ↓
            Final Summary
```

### **Refine**

```text
Page 1 → Summary 1
             ↓
Summary 1 + Page 2 → Summary 2
             ↓
Summary 2 + Page 3 → Summary 3
             ↓
Summary 3 + Page 4 → Final Summary
```

---

# 📐 **Summarization as a Function**

**Let the complete document be:**

$$
D = \{D_1,D_2,\ldots,D_N\}
$$

where each $D_i$ is a document piece.

### **Stuff**

**Conceptually:**

$$
S = f(D_1,D_2,\ldots,D_N)
$$

### **Map-Reduce**

**Map:**

$$
S_i = f(D_i)
$$

Reduce:

$$
S = g(S_1,S_2,\ldots,S_N)
$$

### **Refine**

**Initial:**

$$
S_1 = f(D_1)
$$

**Then:**

$$
S_i = f(S_{i-1},D_i)
$$

This captures the fundamental difference between the three strategies.

---

# 🎓 **What You Should Remember**

```text
1. LLMs have limitations.
2. Private application data is not automatically known by the model.
3. Every model has a context limitation.
4. Independent LLM calls are stateless by default.
5. Models can have a knowledge cutoff.
6. LLMs can hallucinate.
7. Document loaders extract data from source formats.
8. LangChain represents loaded content using Document objects.
9. Document objects commonly contain page_content and metadata.
10. Sending a large document in one prompt can exceed the context limit.
11. Summarization chains help process documents using different strategies.
12. Stuff combines everything into one prompt.
13. Map-Reduce summarizes pieces and then combines the summaries.
14. Refine progressively updates a summary with additional content.
15. The best strategy depends on document size, desired behavior, cost and latency.
```

---

# 🏁 **Final Mental Model**

```text
                  LARGE DOCUMENT
                        │
                        ↓
                DOCUMENT LOADER
                        │
                        ↓
               DOCUMENT OBJECTS
                 ┌──────┴──────┐
                 ↓             ↓
           page_content     metadata
                 │
                 ↓
             LLM LIMITS
                 │
        ┌────────┼────────┐
        ↓        ↓        ↓
     Private   Context  Stateless
       Data    Window    Calls
        │        │        │
        └────────┼────────┘
                 ↓
          SUMMARIZATION
              CHAIN
                 │
       ┌─────────┼──────────┐
       ↓         ↓          ↓
     STUFF    MAP-REDUCE   REFINE
       ↓         ↓          ↓
   One Prompt  Map → Reduce  Iterative
       ↓         ↓          ↓
    Summary   Summary      Summary
```

---

# 📚 **One-Page Revision**

```text
SOURCE
  ↓
PDF / Document
  ↓
LOADER
  ↓
List[Document]
  ↓
 ┌─────────────────────────────────────────┐
 │ Document                                │
 │ ├── page_content → extracted text       │
 │ └── metadata     → source information   │
 └─────────────────────────────────────────┘
  ↓
LONG DOCUMENT
  ↓
CONTEXT LIMIT
  ↓
SUMMARIZATION CHAIN
  ↓
 ┌──────────────┬──────────────┬───────────┐
 ↓              ↓              ↓
STUFF       MAP-REDUCE       REFINE
 ↓              ↓              ↓
All at once  Pieces first   Evolving
 ↓              ↓              ↓
LLM          Combine         Update
 ↓              ↓              ↓
SUMMARY      SUMMARY        SUMMARY
```

---

# 🔗 **Useful Official Resource**

> **LangChain APIs change over time. Use the current official documentation when implementing these patterns in a new project.**

```text
LangChain Documentation
https://docs.langchain.com
```

---

# 🏁 **End Note**

> **The core idea of LangChain document summarization is simple:**

```text
Load the document
      ↓
Convert it into Document objects
      ↓
Avoid exceeding the model's context limits
      ↓
Choose a summarization strategy
      ↓
Generate the summary
```

> **The three classic strategies are easy to remember:**

```text
Stuff
→ Everything together

Map-Reduce
→ Summarize pieces, then combine

Refine
→ Keep improving one summary
```
